Import libraries

In [1]:
import re
import os
import pandas as pd
import numpy as np

In [2]:
import stata_setup
stata_setup.config(os.path.join('/', 'Applications', 'Stata'), 'se')
from pystata import stata


  ___  ____  ____  ____  ____ ®
 /__    /   ____/   /   ____/      18.0
___/   /   /___/   /   /___/       SE—Standard Edition

 Statistics and Data Science       Copyright 1985-2023 StataCorp LLC
                                   StataCorp
                                   4905 Lakeway Drive
                                   College Station, Texas 77845 USA
                                   800-STATA-PC        https://www.stata.com
                                   979-696-4600        stata@stata.com

Stata license: Unlimited-user network, expiring 13 Dec 2025
Serial number: 401809347100
  Licensed to: Diana Tagliaferri
               

Notes:
      1. Unicode is supported; see help unicode_advice.
      2. Maximum number of variables is set to 5,000 but can be increased;
          see help set_maxvar.


Set directory

In [3]:
cd = os.getcwd()
print(cd)

/Users/dianatagliaferri/Library/CloudStorage/OneDrive-LondonBusinessSchool/Documents/Econometrics I/ps2


**Problem 1a**

Read the dataset and perform the required manipulations

In [4]:
# Read the dataset
ps1small = pd.read_csv(os.path.join(cd, 'ps1small.csv'))

# Add the log wage
old_cols = ps1small.columns.tolist()
display(ps1small[ps1small['wage'] <= 0]) # Check that 'wage' is always positive
ps1small['log_wage'] = np.log(ps1small['wage'])
ps1small = ps1small[['log_wage'] + old_cols].copy()

# Create interaction dummies between education and age
display(ps1small[['education', 'age']].dtypes) # Check data types
edu_levels = ps1small['education'].sort_values().unique().tolist()
age_levels = ps1small['age'].sort_values().unique().tolist()
combo_names = [f'{e}{a}' for e in edu_levels for a in age_levels]
edu_age_cat = pd.Categorical(
    ps1small['education'].astype(str) + ps1small['age'].astype(str),
    categories=combo_names
)
dummies = pd.get_dummies(edu_age_cat).astype(int)
dummies.columns = [f'd{col}' for col in dummies.columns]

# Concatenate dummies to the original dataframe
ps1small = pd.concat([ps1small, dummies], axis=1)
ps1small.head()

,wage,education,age


education    int64
age          int64
dtype: object

,log_wage,wage,education,age,d1226,d1227,d1228,d1229,d1230,d1326,...,d1626,d1627,d1628,d1629,d1630,d1726,d1727,d1728,d1729,d1730
0,10.308953,30000,17,30,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1
1,9.680344,16000,12,27,0,1,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,10.680516,43500,17,28,0,0,0,0,0,0,...,0,0,0,0,0,0,0,1,0,0
3,9.798127,18000,12,30,0,0,0,0,1,0,...,0,0,0,0,0,0,0,0,0,0
4,10.126631,25000,17,27,0,0,0,0,0,0,...,0,0,0,0,0,0,1,0,0,0


Regress lwage on the education-age dummies

In [6]:
stata.pdataframe_to_data(ps1small, force=True)

for dummy in dummies.columns:
    edu = dummy[1:3]
    age = dummy[3:]
    stata.run(f'label var {dummy}' + r' "I(\text{edu} = ' + edu + r', \text{age} = ' + age + ')"')

dummies_str = ' '.join(dummies.columns.tolist())

# Build dummy lists split into 3 chunks
parts = np.array_split(dummies.columns.tolist(), 3)
keep1, keep2, keep3 = [" ".join(p) for p in parts]

stata_code = f'''
capture ssc install outreg2
regress log_wage {dummies_str} , nocons

* Export the regression output in three separate fragments 
outreg2 using "p1a_1.tex", replace tex(fragment) label ctitle("lwage") dec(3) keep({keep1}) nonotes noobs nor2
outreg2 using "p1a_2.tex", replace tex(fragment) label ctitle("lwage") dec(3) keep({keep2}) 
outreg2 using "p1a_3.tex", replace tex(fragment) label ctitle("lwage") dec(3) keep({keep3}) nonotes noobs nor2
'''

stata.run(stata_code)

# Fix LaTeX output
subtables = []
for fname in ['p1a_1.tex', 'p1a_2.tex', 'p1a_3.tex']:
    with open(os.path.join(cd, fname), 'r') as f: 
        subtables.append(f.read().replace('VARIABLES', ''))
full_table = r'''
\begin{landscape}
\begin{table}[!htbp]\centering
\caption{Regression of lwage on dummies for education and age}
\label{tab:problem1a}

\begin{subtable}[t]{0.32\textwidth}\centering''' + '\n' + '\n'.join(subtables[0].splitlines()[:-2]) + '\n' + r'''
 & \\
 & \\
 & \\ \hline
 & \\
 & \\
\end{tabular}
\end{subtable}\hfill
\begin{subtable}[t]{0.32\textwidth}\centering''' + '\n' + subtables[1] + '\n' + r'''
\end{subtable}\hfill
\begin{subtable}[t]{0.32\textwidth}\centering''' + '\n' +'\n'.join(subtables[0].splitlines()[:-2]) + '\n' + r'''
 & \\
 & \\
 & \\ \hline
 & \\
 & \\
\end{tabular}
\end{subtable}

\end{table}
\end{landscape}
'''
full_table = re.sub(r'(?<!\$)(I\([^)]*\))(?!\$)', r'$\1$', full_table)
with open(os.path.join(cd, 'p1a.tex'), 'w') as f: f.write(full_table)
print(full_table)


. 
. capture ssc install outreg2

. regress log_wage d1226 d1227 d1228 d1229 d1230 d1326 d1327 d1328 d1329 d1330 
> d1426 d1427 d1428 d1429 d1430 d1526 d1527 d1528 d1529 d1530 d1626 d1627 d1628
>  d1629 d1630 d1726 d1727 d1728 d1729 d1730 , nocons

      Source |       SS           df       MS      Number of obs   =       898
-------------+----------------------------------   F(30, 868)      =   3960.93
       Model |  83238.0973        30  2774.60324   Prob > F        =    0.0000
    Residual |  608.027447       868   .70049245   R-squared       =    0.9927
-------------+----------------------------------   Adj R-squared   =    0.9925
       Total |  83846.1247       898  93.3698493   Root MSE        =    .83695

------------------------------------------------------------------------------
    log_wage | Coefficient  Std. err.      t    P>|t|     [95% conf. interval]
-------------+----------------------------------------------------------------
       d1226 |    9.45132   .0907804  

**Problem 1b**

Export R matrix as LaTeX table

In [40]:
# Obtain lists of unique education and age levels
edu_levels = sorted(set([int(col[1:3]) for col in dummies.columns.tolist()]))
age_levels = sorted(set([int(col[3:5]) for col in dummies.columns.tolist()]))

# Store the column names and indexes of the matrix 
colnames = [f'alpha{edu}{age}' for edu in edu_levels for age in age_levels]
collabels = [fr'$\alpha_{{{edu},{age}}}$' for e in edu_levels for a in age_levels]
idx = {name: i for i, name in enumerate(colnames)}

# Specify anchors 
anchors = {(12,26),(12,27),(13,26),(13,27)}

# Initialize lists of row names and rows
rownames, rows = [], []

for edu in edu_levels:
    for age in age_levels:
        if (edu, age) not in anchors:
            rownames.append(fr'$\alpha_{{{edu},{age}}}$' )
            row = np.zeros(len(colnames))
            row[idx[f'alpha{edu}{age}']] = 1
            row[idx['alpha1226']] = -1 + (edu-12) + (age-26) - (edu-12)*(age-26)
            row[idx['alpha1327']] = - (edu-12) + (edu-12)*(age-26)
            row[idx['alpha1227']] = - (age-26) + (edu-12)*(age-26)
            row[idx['alpha1327']] = - (edu-12)*(age-26)
            rows.append(row)

R=pd.DataFrame(np.asarray(rows, dtype=int), index=rownames, columns=collabels)
R.to_latex(
    buf="p1b.tex",
    index=False, header=True, escape=False,
    column_format='l' + 'c'*len(colnames),  # left row-label + numeric columns
)

Perform F test in STATA

In [43]:
# Build the joint restriction string in Python
tests = []
anchors = {(12,26), (12,27), (13,26), (13,27)}
for edu in edu_levels:
    for age in age_levels:
        if (edu, age) in anchors:
            continue
        r = edu - 12
        s = age - 26
        c1226 = -1 + (edu-12) + (age-26) - (edu-12)*(age-26)
        c1326 = -(edu-12) + (edu-12)*(age-26)
        c1227 = -(age-26) + (edu-12)*(age-26)
        c1327 = -(edu-12)*(age-26)
        expr = f'( d{edu}{age} + {c1226}*d1226 + {c1326}*d1326 + {c1227}*d1227 + {c1327}*d1327 = 0 )'
        tests.append(expr)

test_stmt = " ".join(tests)

# Run the unrestricted regression and F test in STATA
stata.pdataframe_to_data(ps1small, force=True)

stata.run(fr'''
* Unrestricted regression
regress log_wage {dummies_str} , nocons

* F test of all 26 restrictions:
test {test_stmt}
''')



. 
. * Unrestricted regression
. regress log_wage d1226 d1227 d1228 d1229 d1230 d1326 d1327 d1328 d1329 d1330 
> d1426 d1427 d1428 d1429 d1430 d1526 d1527 d1528 d1529 d1530 d1626 d1627 d1628
>  d1629 d1630 d1726 d1727 d1728 d1729 d1730 , nocons

      Source |       SS           df       MS      Number of obs   =       898
-------------+----------------------------------   F(30, 868)      =   3960.93
       Model |  83238.0973        30  2774.60324   Prob > F        =    0.0000
    Residual |  608.027447       868   .70049245   R-squared       =    0.9927
-------------+----------------------------------   Adj R-squared   =    0.9925
       Total |  83846.1247       898  93.3698493   Root MSE        =    .83695

------------------------------------------------------------------------------
    log_wage | Coefficient  Std. err.      t    P>|t|     [95% conf. interval]
-------------+----------------------------------------------------------------
       d1226 |    9.45132   .0907804   10